In [2]:
import numpy as np
import pandas as pd
from datasets import Dataset
from sklearn.metrics import f1_score, classification_report
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

d:\Ai\miniconda3\envs\first\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
train = pd.read_csv("../data/train_data.csv")
test = pd.read_csv("../data/test_data.csv")

C:\Users\IT CITY\AppData\Local\Temp\ipykernel_19856\3270014727.py:1: DtypeWarning: Columns (0: vote) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv("../data/train_data.csv")


In [4]:
name = "microsoft/deberta-v3-base"
num_samples = 120000
max_length = 192

train = train.dropna(subset=["overall"])
if len(train) > num_samples:
    train = train.sample(num_samples, random_state=42)

train["label"] = train["overall"].astype(int) - 1
train["text"] = train["summary"].fillna("") + ". " + train["reviewText"].fillna("")
test["text"] = test["summary"].fillna("") + ". " + test["reviewText"].fillna("")

tokenizer = AutoTokenizer.from_pretrained(name)

In [7]:
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=max_length)


ds = Dataset.from_pandas(train[["text", "label"]], preserve_index=False)
ds = ds.train_test_split(test_size=0.05, seed=42)
ds = ds.map(tokenize, batched=True, remove_columns=["text"])

test_ds = Dataset.from_pandas(test[["text"]], preserve_index=False)
test_ds = test_ds.map(tokenize, batched=True, remove_columns=["text"])

import torch

model = AutoModelForSequenceClassification.from_pretrained(
    name, num_labels=5, dtype=torch.float32
)


def compute_metrics(p):
    pred = np.argmax(p.predictions, axis=1)
    return {
        "accuracy": (pred == p.label_ids).mean(),
        "micro_f1": f1_score(p.label_ids, pred, average="micro"),
    }

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 1443.24it/s]
[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    

In [8]:

args = TrainingArguments(
    output_dir="/content/drive/MyDrive/out",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    warmup_steps=0.06,
    weight_decay=0.01,
    fp16=True,
    eval_strategy="steps",
    eval_steps=2000,
    save_strategy="steps",
    save_steps=2000,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=200,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

val_out = trainer.predict(ds["test"])
val_pred = np.argmax(val_out.predictions, axis=1)
print("micro f1:", f1_score(ds["test"]["label"], val_pred, average="micro"))
print(classification_report(ds["test"]["label"], val_pred))

test_out = trainer.predict(test_ds)
test_pred = np.argmax(test_out.predictions, axis=1) + 1

submission = pd.DataFrame({"predicted": test_pred})
submission.to_csv("q2_submission.csv", index=False)
print(submission.head())
print(len(submission), len(test))

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Step,Training Loss,Validation Loss,Accuracy,Micro F1
2000,1.194452,0.591249,0.765167,0.765167
4000,1.042681,0.575471,0.775833,0.775833
6000,0.978165,0.582091,0.775500,0.775500
7126,0.976460,0.572667,0.780833,0.780833


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.88it/s]


micro f1: 0.7808333333333334
              precision    recall  f1-score   support

           0       0.72      0.80      0.76       612
           1       0.52      0.44      0.48       439
           2       0.58      0.55      0.56       567
           3       0.65      0.60      0.62      1094
           4       0.89      0.92      0.91      3288

    accuracy                           0.78      6000
   macro avg       0.67      0.66      0.67      6000
weighted avg       0.77      0.78      0.78      6000



   predicted
0          1
1          1
2          1
3          1
4          1
20000 20000
